In [1]:
!pip install HEBO -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 795.9/795.9 kB 10.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.7/72.7 kB 4.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.4/60.4 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.7/114.7 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 249.1/249.1 kB 14.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.8/3.8 MB 14.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 277.7/277.7 kB 16.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.3/17.3 MB 93.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 98.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 80.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 40.6 MB/s eta 0:00:00
 

In [2]:
!pip install botorch -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 779.9/779.9 kB 25.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 756.0/756.0 kB 33.7 MB/s eta 0:00:00


In [3]:
# how to optimize with hebo: https://github.com/huawei-noah/HEBO/blob/master/HEBO/doc/source/optimisation.ipynb
from hebo.optimizers.hebo import HEBO
from hebo.design_space.design_space import DesignSpace


In [ ]:
from botorch.test_functions import Shekel


In [5]:
import torch

dtype = torch.double

In [ ]:
from tqdm import tqdm


def test_hebo(budget, init_n=4, q=1):
    # setup problem (fast)
    s4_problem = Shekel(noise_std=None, negate=True)  # assume maximization problem
    lbs = s4_problem.bounds[0, :]
    ubs = s4_problem.bounds[1, :]
    params = [
        {'name': f'x{i}', 'type': 'num', 'lb': lbi, 'ub': ubi} 
        for i, lbi, ubi in zip(range(s4_problem.dim), lbs.detach().numpy(), ubs.detach().numpy())]
    hebo_design_space = DesignSpace().parse(params)
    # init optimizer
    optimizer = HEBO(hebo_design_space, rand_sample=init_n)
    # optimize
    best_results = []
    trials_spend = 0
    with tqdm(total=budget, leave=False)as pbar:
        while trials_spend < budget:
            cand = optimizer.suggest(n_suggestions=1)
            x = torch.from_numpy(cand.iloc[-1].to_numpy())
            y = s4_problem(x)
            optimizer.observe(cand, -1 * y.detach().numpy().reshape(-1))  # negate for optimizing for minimization problem

            # current_best_max = -optimizer.y.min()
            if best_results:
                best_results.append(max(best_results[-1], y.item())) 
            else:
                best_results.append(y.item())
            pbar.update(1)
            trials_spend += 1

    return best_results


def conduct_experiment(n_repeats, budget=100, init_n=1, q=1):
    all_res = []
    for i in tqdm(range(n_repeats), total=n_repeats):
        res = test_hebo(budget=budget, init_n=init_n, q=q)
        all_res.append(res)
    return all_res



In [7]:
vanilla_hebo_res = conduct_experiment(n_repeats=30, budget=100, init_n=4, q=1)  # 1 repeat - 9 min,

  0%|          | 0/100 [00:00<?, ?it/s]

jitter = 1e-07
jitter = 1e-06
jitter = 9.999999999999999e-06
jitter = 9.999999999999999e-05
jitter = 0.001
jitter = 0.01
jitter = 0.1
jitter = 1.0
jitter = 10.0
jitter is too large, give up fitting GP
jitter = 1e-07
jitter = 1e-06
jitter = 9.999999999999999e-06
jitter = 9.999999999999999e-05
jitter = 0.001
jitter = 0.01
jitter = 0.1
jitter = 1.0
jitter = 10.0
jitter is too large, give up fitting GP
jitter = 1e-07
jitter = 1e-06
jitter = 9.999999999999999e-06
jitter = 9.999999999999999e-05
jitter = 0.001
jitter = 0.01
jitter = 0.1
jitter = 1.0
jitter = 10.0
jitter is too large, give up fitting GP
jitter = 1e-07
jitter = 1e-06
jitter = 9.999999999999999e-06
jitter = 9.999999999999999e-05
jitter = 0.001
jitter = 0.01
jitter = 0.1
jitter = 1.0
jitter = 10.0
jitter is too large, give up fitting GP
jitter = 1e-07
jitter = 1e-06
jitter = 9.999999999999999e-06
jitter = 9.999999999999999e-05
jitter = 0.001
jitter = 0.01
jitter = 0.1
jitter = 1.0
jitter = 10.0
jitter is too large, give up fittin


  5%|▌         | 5/100 [00:05<01:51,  1.18s/it]

jitter = 1e-07
jitter = 1e-06
jitter = 9.999999999999999e-06
jitter = 9.999999999999999e-05
jitter = 0.001
jitter = 0.01
jitter = 0.1
jitter = 1.0
jitter = 10.0
jitter is too large, output random predictions
jitter = 1e-07
jitter = 1e-06
jitter = 9.999999999999999e-06
jitter = 9.999999999999999e-05
jitter = 0.001
jitter = 0.01
jitter = 0.1
jitter = 1.0
jitter = 10.0
jitter is too large, output random predictions
jitter = 1e-07
jitter = 1e-06
jitter = 9.999999999999999e-06
jitter = 9.999999999999999e-05
jitter = 0.001
jitter = 0.01
jitter = 0.1
jitter = 1.0
jitter = 10.0
jitter is too large, output random predictions



  5%|▌         | 5/100 [00:02<00:56,  1.68it/s]

jitter = 1e-07


... (7 outputs left out)

jitter = 1e-07
jitter = 1e-07
jitter = 1e-07
jitter = 1e-07
jitter = 1e-07
jitter = 1e-07
jitter = 1e-07
jitter = 1e-07
jitter = 1e-07
jitter = 1e-07
jitter = 1e-07
jitter = 1e-07
jitter = 1e-07



  6%|▌         | 6/100 [00:06<01:52,  1.20s/it]

jitter = 1e-07
jitter = 1e-07
jitter = 1e-07
jitter = 1e-07
jitter = 1e-07
jitter = 1e-07
jitter = 1e-07
jitter = 1e-07
jitter = 1e-07
jitter = 1e-07



100%|██████████| 30/30 [2:27:12<00:00, 294.43s/it]


In [8]:
# vanilla_hebo_res

In [ ]:
import numpy as np


np.savetxt("/kaggle/working/vanilla_hebo_res_S4.txt", np.array(vanilla_hebo_res))